# Silver Order Items

Clean and transform order items data from bronze to silver layer with data quality checks: clean data, validate keys, and remove duplicates.

In [0]:
%run "/Users/jaygupta9584@gmail.com/Intelligent Customer 360 & Support Analytics Platform/Common/CommonHelper"

In [0]:
%run "/Users/jaygupta9584@gmail.com/Intelligent Customer 360 & Support Analytics Platform/Common/SilverLoader"

In [0]:
silver_table_name = get_table_name("order_items", "silver")
bronze_table_name = get_table_name("order_items", "bronze")

In [0]:
# spark.read.table(bronze_table_name).display()

In [0]:
from pyspark.sql import functions as F

# --- Transform function for silver order_items ---
def transform_order_items(bronze_df):
    silver_df = (bronze_df
        .select(
            F.col("order_item_id"),
            F.col("order_id"),
            F.col("product_id"),
            F.col("quantity"),
            F.col("unit_price"),
            F.col("discount"),
            F.col("created_at"),
            F.col("_bronze_ingestion_ts")
        )
    )
    silver_df = validate_keys(silver_df, ["order_item_id"])
    silver_df = remove_duplicates(silver_df, ["order_item_id"], order_col="created_at")
    silver_df = drop_bronze_metadata(silver_df)
    return silver_df

result = incremental_silver_load(
    bronze_table_name=bronze_table_name,
    silver_table_name=silver_table_name,
    key_columns=["order_item_id"],
    transform_fn=transform_order_items,
    update_condition=None
)

print(result)

In [0]:
display(spark.read.table(silver_table_name))